In [ ]:
import time
import requests
from bs4 import BeautifulSoup as bsSoup
import warnings
import pandas as pd
import os

**a.** Design and implement a solution to crawl the publication title, publication venue, publication year, author list and of every unique publication record on the target website. **[15%]**

In [ ]:
publicationsIndex = "https://sitescrape.awh.durham.ac.uk/comp42315/publicationfull_year_characteranimation.htm"

# function that scrapes the links of all the sites where unique publications can be
def getLinks (url: str) -> list :
    sitePrefix = "https://sitescrape.awh.durham.ac.uk/comp42315/"

    if (not isinstance(url, str)) :
        print("The URL needs to be a string!")
        return []

    # wait a little so we don't overload the server
    time.sleep(2)
    publicationsIndex = requests.get(url, verify = False)

    if (publicationsIndex.status_code != 200) :
        print (f"Error; status code returned: {publicationsIndex.status_code}")
        return []

    publicationsIndexSoup = bsSoup(publicationsIndex.content, "html.parser").body

    if (publicationsIndexSoup == None) :
        print ("Nothin to parse on the site")
        return []

    publicationLinksA = publicationsIndexSoup.find("p", class_ = "TextOption").find_all("a")

    if (publicationLinksA == None) :
        print ("No links found")
        return []
    
    links = [sitePrefix + n.get("href") for n in publicationLinksA]

    if (links[0] == None) :
        print("No links found")
        return []

    return links

In [ ]:
categoriesUrls = [publicationsIndex] + getLinks(publicationsIndex)

# this funcion scrapes publication title, publication type (only bit that can't be acquired on the publication's page) and the link of the webpage - this link is needed to acquire the precise impact score and citation count - it also has the remaining required information: year, venue, list of authors; it returns a dictionary in a format 
# key: publication type -> value: [title, page link]
def scrapePublicationWebpageLinks (urls: list, replace : str) -> dict :
    scrapedData = {}
    seenTitles = set()   # ensure we only get each title once
    sitePrefix = "https://sitescrape.awh.durham.ac.uk/comp42315/"

    for url in urls :
        uniqueValuesOnPage = []
        time.sleep(0)
        url = url.replace("year", replace)

        page = requests.get(url, verify = False)

        if (page.status_code != 200) :
            print(f"Failed for link {url}, status code: {page.status_code}, continuing execution for the remaining links")
            continue

        pageSoup = bsSoup(page.content, "html.parser").body

        if (pageSoup == None) :
            print (f"Found nothing to parse on site {url}, continuing execution for the remaining links")
            continue

        pageSoupDiv = pageSoup.find("div", id = "divBackground")

        if (pageSoupDiv == None) :
            print (f"Couldn't find {replace} on the page {url}, continuing execution for the remaining links")
            continue

        pageSoupP = pageSoupDiv.find_all("p", class_ = "TextOption")

        if (pageSoupP == None) :
            print (f"Couldn't find {replace} on the page {url}, continuing execution for the remaining links")
            continue

        paragraphWithInfo = pageSoupP [2]

        valueTags = paragraphWithInfo.find_all("a")

        uniqueValuesOnPage = [n.text for n in valueTags]

        if (len(uniqueValuesOnPage) == 0) :
            print (f"Couldn't find {replace} on the page {url}, continuing execution for the remaining links")
            continue

        for value in uniqueValuesOnPage :
            currentH2 = pageSoup.find("h2", id = value)

            if (currentH2 == None) :
                continue

            if (value not in scrapedData) :
                scrapedData [value] = []

            div = currentH2.findNext()

            publicationsWithThisInfo = div.find_all("div", class_ = "w3-cell-row")

            if (publicationsWithThisInfo == None) :
                continue

            for publication in publicationsWithThisInfo :
                publicationText = publication.find("div", class_ = "w3-container w3-cell w3-mobile w3-cell-middle")

                title = publicationText.text.split("by")[0].strip()

                if (title in seenTitles) :
                    continue

                seenTitles.add(title)

                span = publicationText.find("span")

                publicationLink = span.find("a").get("href")

                listToAdd = [title, sitePrefix + publicationLink]
                scrapedData[value].append(listToAdd)

    return scrapedData

initialScrape = scrapePublicationWebpageLinks(categoriesUrls, "type")

# extract the urls of the publications' sites
publicationUrls : str = []

for k in initialScrape :
    for n in initialScrape[k] :
        publicationUrls.append(n[1])

# this funciton uses the links scraped by the previous funcion and it scrapes all the information that is needed 
def scrapeAdditionalInformation (urls: list) -> dict :
    addtionalInfo = {}

    # find the title, find number of citations and the impact score
    for url in urls :
        time.sleep(0)
        
        infoPage = requests.get(url, verify = False)

        if (infoPage.status_code != 200) :
            print(f"Failed for link {url}, status code: {infoPage.status_code}, continuing execution for the remaining links")
            continue

        infoPageSoup = bsSoup(infoPage.content, "html.parser").body

        if (infoPageSoup == None) :
            print (f"Found nothing to parse on site {url}, continuing execution for the remaining links")
            continue

        infoPageSoupDiv = infoPageSoup.find_all("div", style = "margin-left: var(--size-marginleft)")[1]

        if (infoPageSoupDiv == None) :
            print (f"Found no divs maching the criteria on site {url}, continuing execution for the remaining links")
            continue 

        infoPageSoupP = infoPageSoupDiv.find("p")
        
        title = infoPageSoupP.text.split("by")[0].strip()

        # so that is going to have the number of citations and impact score IF the publication had any
        infoPageSoupDiv2 = infoPageSoupDiv.find_all("div")[2].text
        # then add to dictionary
        addtionalInfo[title] = infoPageSoupP.text + infoPageSoupDiv2

    return addtionalInfo
 

supportingInfo = scrapeAdditionalInformation(publicationUrls)

In [ ]:
# those 2 funcitons will combine the results from the two dictioraies acquired in the previous step
def dictConverter (dataDict : dict) -> dict :
    convertedDict = {}

    for k in dataDict :
        for n in dataDict[k] :
            convertedDict[n[0]] = k

    return convertedDict

def dictCombine (dict1 : dict, dict2 : dict) -> dict :
    combined = {}

    if (len(dict1) != len(dict2)) :
        return {}

    for k in dict1 :
        if (k not in dict1 or k not in dict2) :
            continue

        combined[k] = [dict1[k], dict2[k]]

    return combined

typeDictClean = dictConverter(initialScrape)
rawDataFinal = dictCombine(typeDictClean, supportingInfo)   # this is the final scraped data that I will be working with

**b.**	This information should then be stored and displayed in an appropriate format. Displayed to the user should be: publication title, publication venue, year and author list of every unique publication record on the target website. **[10%]** 

In [ ]:
# this part of the code cleans up the data and stores it into a dictionary and a pandas dataframe

publications = {}
authorsListConcat : str = []
publicationsData : pd.DataFrame

for k in rawDataFinal :
    title : str = k
    allData : str = rawDataFinal[k][0] + "REST " + rawDataFinal[k][1]
    remainder = ""

    initialClean = allData.translate({ord(i): None for i in '\t\r\n'})
    splitAt = initialClean.find("REST")

    typeOfPublication = initialClean[:splitAt]
    remainder = initialClean[splitAt + 4:]

    # remove title
    splitAt = initialClean.find("by")
    remainder = initialClean[splitAt + 3:]

    # find authors
    splitAt = remainder.find(" in ")
    authors = remainder[:splitAt]

    year = int(remainder[splitAt + 4:splitAt + 8])
    remainder = remainder[splitAt + 8:]

    authorsListConcat.append(authors)

    # splitting authors into a list of names
    authorList1 = authors.split(",")
    authorList2 = authorList1[-1].split("and")
    authorList1Strip = [n.strip() for n in authorList1]
    authorList2Strip = [n.strip() for n in authorList2]

    authorList = authorList1Strip[:-1] + authorList2Strip

    # split into publication venue, citations and imact factor if it got one
    splitAt = remainder.find("Citation")
    publicationVenue = remainder[:splitAt]
    remainder = remainder[splitAt + len("Citation: "):]

    split = remainder.split("##")
    citations = int(split[0])

    remainder = split[1]

    impactFactorRaw = remainder.split(": ")
    impactFactor = 0

    if (len(impactFactorRaw) > 1) :
        impactFactor = float(impactFactorRaw[1][:-1])

    publications[title] = [authorList, year, publicationVenue, typeOfPublication, citations, impactFactor]
    

In [ ]:
_data = {"title" : [k for k in publications], 
         "authors" : [n for n in authorsListConcat], 
         "year" : [publications[k][1] for k in publications], 
         "publication venue" : [publications[k][2] for k in publications],
         "type" : [publications[k][3] for k in publications],
         "number of citations" : [publications[k][4] for k in publications],
         "impact" : [publications[k][5] for k in publications]}
         
publicationsData = pd.DataFrame(data = _data)

#publicationsData.head()

In [ ]:
# following 3 functions allow for displaying of data: printing to the console formatted data, exporting to .csv and .txt
def printScrapedDataToConsole () -> None :
    s = 1
    for k in publications :
        print ("---")
        print (s)
        print (f"Title: {k}\n")
        print("Authors: ")

        count = 0
        for n in publications[k][0] :
            if (len(publications[k][0]) == 1 and n == "") :
                print("No authors found")
            elif (count == len(publications[k][0]) - 1) :
                print("and " + n + ".") 
            else :
                print(n + ", ")
            count += 1
            
        print (f"\nYear published: {publications[k][1]}")
        print (f"Publication venue: {publications[k][2]}")
        print (f"Type: {publications[k][3]}")
        print (f"Number of citations: {publications[k][4]}")
        print (f"Impact Factor: {publications[k][5]}")
        print ("---\n")

        s += 1

In [ ]:
def exportToCsv (fileName : str, _data : pd.DataFrame) -> None :
    if (_data.empty) :
        print ("Provide correct Data Frame")
        return

    if (not isinstance(fileName, str)) :
        fileName = str(fileName)
        
    if (os.path.exists(f"./{fileName}.csv")) :
        warnings.warn ("This file already exists")
        return
    
    if (fileName[-4:] != ".csv") :
        fileName = fileName + ".csv"
    
    _data.to_csv(f"{fileName}", index = False)

def exportToTxt (fileName : str, _data : dict) -> None :
    if (not isinstance(fileName, str)) :
        fileName = str(fileName)

    if (os.path.exists(f"./{fileName}.txt")) :
        warnings.warn ("This file already exists")
        return

    if (fileName[-4:] != ".txt") :
        fileName = fileName + ".txt"

    with open (fileName, "w") as f :
        for k in _data :
            f.write(f"{k}\n")

            f.write("by")
            s = 0
            for n in _data[k][0] :
                if (s == len(_data) - 1) :
                    f.write(n)
                else :
                    f.write(n + ", ")
                s += 1

            f.write(f". {_data[k][1]}, {_data[k][2]}, {_data[k][3]}, number of citations: {_data[k][4]}, impact factor: {_data[k][5]}\n")
            f.write("\n")

**c.**	The records should be manipulatable: at minimum you should be able to display sorted information according to descending year values, descending number of author values, the titles from A to Z, and finally by the impact of the papers.  **[10%]** 

Note, this does not require user interaction, this can and should be completed via pre defined functions. 

In [ ]:
def sortDataFrame (_data : pd.DataFrame, sortBy : str, asc : bool = True) -> None :
    df = _data.sort_values(by = [sortBy], ascending = asc)
    return df

#exportToTxt("myTextFile", publications)

**d.** Good programming practice: marks will be given for good Python programming practice **[10%]**

- Defensive progreamming: checking the response codes after an https request, checking that the links are in correct format, checking for None before parsing or searching via beautiful soup
- Making the function wait a few seconds between each request so the server is not overloeaded
- Using descriptive and unique variable names
- Where adequate adding comments to explain the code - at the start of the funciton or where the code is complicated
- Encapsulating code into functions for reusablity and to limit the number of global variables
- Adding type hints to the function params and outputs to improve usablility 
- Using list comprehensions and built-in methods to achieve Pythonic code

**e.** Explain your design and highlight any features in this question’s report part of your Jupyter Notebook in no more than 300 words. Reflect upon the importance of well structured HTML. **[5%]**

I'd say although this webpage's HTML was structured in a weird way, with non-descriptive classes/ ids, it was consistent in it's design. Well structured, non-dynamic HTML, would imporove the websites accessibility by making it easier for screen readers. 

Question 2

In [ ]:
import random as rd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.feature_selection import r_regression, SelectKBest, SequentialFeatureSelector
from sklearn.neighbors import KNeighborsRegressor

droneData = pd.read_csv("drone.csv")
droneData.head()

# features and target

In [ ]:
def takeBootstrapSample (inBagProportion: float, df: pd.DataFrame) -> tuple :
    rowIndicies = []
    for i in range(1,droneData.shape[0] + 1) :
        rowIndicies.append(i)

    trainingSubsetRows = rd.sample(rowIndicies, round(len(rowIndicies) * inBagProportion))
    # converting to sets
    allRowsS = set (rowIndicies)
    trainingRowsS = set (trainingSubsetRows)
    validationSubsetRowsS = allRowsS - trainingRowsS
    # now get the actual rows from the dataframe
    trainingRowsBool = []
    validationRowsBool = []
    for i in rowIndicies :
        if i in trainingRowsS :
            trainingRowsBool.append(True)
        else :
            trainingRowsBool.append(False)

        if i in validationSubsetRowsS :
            validationRowsBool.append(True)
        else :
            validationRowsBool.append(False)

    trainingData = df.iloc[trainingRowsBool]
    validationData = df.iloc[validationRowsBool]

    return (trainingData, validationData)

bootstrapSample = takeBootstrapSample(0.75, droneData)

In [ ]:
trainingDataset = bootstrapSample[0]

# this returns the number of NaNs for each column 
#print (trainingDataset.isna().sum())
# this returns a specific row that contains both NaNs: 3107
#print(trainingDataset[trainingDataset["load"].isna()])
# I can just remove it as it is a single row in more than 4000 observations
trainingDatasetRemNAs = trainingDataset.drop(3107)

# perhaps more data cleaning is necessary and do it here


# split the dataframe into the predictors and response
allPredictors = trainingDatasetRemNAs.loc[:, trainingDataset.columns != "warning"]
response = trainingDatasetRemNAs["warning"]

# convert to numpy array
allPredictorsNp = allPredictors.to_numpy()
responseNp = response.to_numpy()


# calculate the Pearson correlation coefficients via the r-regression
scores = r_regression(allPredictorsNp, responseNp)
print(scores)
# ok, I am assuming I choose the predictors with the higest corr coef
# ok so this is not a standalone feature select method - it is a scoring method used in feature selection so for e.g. together with select K best
# return indexes of 3 highest outputs
# ok just take it out manually lol

# select k best approach

# apparently that works with a data frame directly
selector = SelectKBest(r_regression, k = 3)
selectKBestRawOut = selector.fit_transform(allPredictors, response)

colIds = selector.get_support(indices = True)

skbOutDf = trainingDataset.iloc[:, colIds]
print(skbOutDf.columns)

# using sequential feature selection
knn = KNeighborsRegressor(3)
sequentialFeatureSelectRawOut = SequentialFeatureSelector(knn, n_features_to_select = 3).fit(allPredictors, response)
cols = sequentialFeatureSelectRawOut.get_support(indices = True)
sfsOutDf = trainingDataset.iloc[:, cols]
print (sfsOutDf.columns)

Data binning (applied to the training set only) In this task you are requested to carry out data binning for the features of each data set obtained at the output of the previous part.

i. In particular, the values of each feature should be divided into 5 intervals with the smallest and largest values being the start and the end of the interval being partitioned.
ii. After that each value is to be replaced with the interval it belongs.
iii. Please implement two versions of binning for each data frame.
iv. In the first part just apply an ordinary binning. In the second part, the binning should be preceded by removal of outliers. The removal of outliers should be based on the interquartile range (IQR) method. Recall that, according to this method, a value is an outlier is it is smaller than the first quartile by more than 1.5IQR or greater than the third quartile by more than 1.5IQR.
v. Thus, the output of this step will consist of four datasets obtained by applying two ways of binning for each dataset obtained as the output of the previous step.
vi. In the text part (part f) of this task, please briefly discuss positive and negative effects of binning and removal of outliers in the context of ML.

In [ ]:
def diagnosticPlots () :
    plt.scatter(skbOutDf["current_a"], skbOutDf["current_filtered_a"])
    plt.title("scatterplot of current_filterd_a vs current_a")
    plt.show()
    print ("Person corr matrix for current_a and current_filtered_a - very strong positive correlation")
    print (np.corrcoef(skbOutDf["current_a"], skbOutDf["current_filtered_a"]))

    plt.scatter(sfsOutDf["voltage_filtered_v"], sfsOutDf["current_a"])
    plt.title("scatterplot of voltage_filtered_v vs current_a")
    plt.show()
    print (np.corrcoef(sfsOutDf["voltage_filtered_v"], sfsOutDf["current_a"]))

    fig, axs = plt.subplots(2)
    fig.suptitle("current_a")
    axs[0].scatter(skbOutDf.index, skbOutDf["current_a"], alpha = 0.4, s = 9)
    axs[1].hist(skbOutDf["current_a"], bins = 15)

    fig1, axs = plt.subplots(2)
    fig1.suptitle("current_filtered_a")
    axs[0].scatter(skbOutDf.index, skbOutDf["current_filtered_a"], alpha = 0.4, s = 9)
    axs[1].hist(skbOutDf["current_filtered_a"], bins = 15)

    fig2, axs = plt.subplots(2)
    fig2.suptitle("scale")
    axs[0].scatter(skbOutDf.index, skbOutDf["scale"], alpha = 0.4, s = 9)
    axs[1].hist(skbOutDf["scale"], bins = 6)

    fig3, axs = plt.subplots(2)
    fig3.suptitle("remaining")
    axs[0].scatter(sfsOutDf.index, sfsOutDf["remaining"], alpha = 0.4, s = 9)
    axs[1].hist(sfsOutDf["remaining"], bins = 6)

    fig4, axs = plt.subplots(2)
    fig4.suptitle("voltage_filtered_v")
    axs[0].scatter(sfsOutDf.index, sfsOutDf["voltage_filtered_v"], alpha = 0.4, s = 9)
    axs[1].hist(sfsOutDf["voltage_filtered_v"], bins = 12)


diagnosticPlots()
# oh god almighty
currentACut = pd.cut(skbOutDf["current_a"], 5)
currentFilteredACut = pd.cut(skbOutDf["current_filtered_a"], 5)
scaleCut = pd.cut(skbOutDf["scale"], 5)
remainingCut = pd.cut(sfsOutDf["remaining"], 5)
voltageCut = pd.cut(sfsOutDf["voltage_filtered_v"], 5)
print (voltageCut)
print (remainingCut)

# print (skbOutDf.columns)
# print (skbOutDf.shape)
# print (skbOutDf)